# Workout Class Classifier - Bidirectional LSTM

This notebook trains a Bidirectional LSTM (BiLSTM) classifier to predict workout classes from pose sequences.

The model:
- Takes sequences of shape (15, 12, 3) - 15 frames, 12 landmarks, 3 coordinates
- Reshapes to (15, 36) - 15 timesteps, 36 features per timestep (12 landmarks × 3 coords)
- Uses Bidirectional LSTM to capture temporal patterns in both directions
- Predicts workout classes using a dense output layer

In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Check TensorFlow version
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

ModuleNotFoundError: No module named 'pandas'

## Load Training Data

In [ ]:
# Load training data from .npz file
data_path = Path('../../Data/output/training_data.npz')
metadata_path = Path('../../Data/output/training_data_metadata.json')

# Load metadata
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print("Dataset Metadata:")
print(f"  Number of samples: {metadata['n_samples']}")
print(f"  Number of classes: {metadata['n_classes']}")
print(f"  Sequence length: {metadata['sequence_length']}")
print(f"  Number of landmarks: {metadata['n_landmarks']}")
print(f"  Number of coordinates: {metadata['n_coords']}")
print(f"  Feature shape: {metadata['feature_shape']}")
print(f"\nClass names: {metadata['class_names']}")
print(f"\nClass distribution:")
for cls, count in metadata['class_distribution'].items():
    print(f"  {cls}: {count}")

# Load the .npz file
data = np.load(data_path, allow_pickle=True)

# Load data
X = data['X']  # Shape: (n_samples, 15, 33, 3)
y_encoded = data['y']  # Encoded labels (integers)
y_raw = data['y_raw']  # Raw class names
class_names = data['class_names']  # Class names array

print(f"\nLoaded data shapes:")
print(f"  X shape: {X.shape}")
print(f"  y_encoded shape: {y_encoded.shape}")
print(f"  Number of classes: {len(class_names)}")

## Preprocess Data

Reshape sequences from (15, 12, 3) to (15, 36) for LSTM input - each timestep contains flattened landmark coordinates.

In [ ]:
# Reshape sequences: (n_samples, 15, 12, 3) -> (n_samples, 15, 36)
# Each timestep (frame) contains 12 landmarks × 3 coordinates = 36 features
X_reshaped = X.reshape(X.shape[0], X.shape[1], -1)
print(f"Reshaped X shape: {X_reshaped.shape}")
print(f"  - {X_reshaped.shape[0]} samples")
print(f"  - {X_reshaped.shape[1]} timesteps (frames)")
print(f"  - {X_reshaped.shape[2]} features per timestep")

# Check for NaN or infinite values
print(f"\nData quality check:")
print(f"  NaN values in X: {np.isnan(X_reshaped).sum()}")
print(f"  Infinite values in X: {np.isinf(X_reshaped).sum()}")

# Replace any NaN or infinite values with 0
X_reshaped = np.nan_to_num(X_reshaped, nan=0.0, posinf=0.0, neginf=0.0)

# Normalize the data (important for neural networks)
# Calculate mean and std from training data
X_mean = np.mean(X_reshaped, axis=(0, 1), keepdims=True)
X_std = np.std(X_reshaped, axis=(0, 1), keepdims=True) + 1e-8  # Add small epsilon to avoid division by zero

# Normalize
X_normalized = (X_reshaped - X_mean) / X_std
print(f"\nData normalized: mean={X_mean.mean():.4f}, std={X_std.mean():.4f}")

# Convert labels to categorical (one-hot encoding) for neural network
y_categorical = keras.utils.to_categorical(y_encoded, num_classes=len(class_names))
print(f"y_categorical shape: {y_categorical.shape}")

## Split Data into Train and Test Sets

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train_cat, y_test_cat, y_train_encoded, y_test_encoded = train_test_split(
    X_normalized, y_categorical, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # Ensure balanced split across classes
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Sequence shape: {X_train.shape[1:]} (timesteps, features)")
print(f"Number of classes: {len(class_names)}")

## Build BiLSTM Model

Create a Bidirectional LSTM model with:
- Input layer for sequences
- Bidirectional LSTM layers to capture temporal patterns
- Dropout for regularization
- Dense layers for classification

In [ ]:
# Model parameters
n_timesteps = X_train.shape[1]  # 15 frames
n_features = X_train.shape[2]    # 36 features per timestep (12 landmarks × 3 coords)
n_classes = len(class_names)     # 22 workout classes

# Build the model
model = models.Sequential([
    # Input layer
    layers.Input(shape=(n_timesteps, n_features)),
    
    # First Bidirectional LSTM layer
    layers.Bidirectional(
        layers.LSTM(256, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        name='bilstm_1'
    ),
    
    layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        name='bilstm_2'
    ),

    layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        name='bilstm_3'
    ),
    
    # Second Bidirectional LSTM layer
    layers.Bidirectional(
        layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.2),
        name='bilstm_4'
    ),
    
    # Dense layer with dropout
    layers.Dense(64, activation='relu', name='dense_1'),
    layers.Dropout(0.3, name='dropout_1'),
    
    # Output layer
    layers.Dense(n_classes, activation='softmax', name='output')
])

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
model.summary()

## Train the Model

Train the BiLSTM model with early stopping and model checkpointing.

In [ ]:
# Define callbacks
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=2e-6,
    verbose=1
)

# Create models directory if it doesn't exist
models_dir = Path('models')
models_dir.mkdir(exist_ok=True)

model_checkpoint = callbacks.ModelCheckpoint(
    filepath=str(models_dir / 'bilstm_workout_classifier_best.keras'),
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Training parameters
batch_size = 64
epochs = 100

print("Training BiLSTM model...")
print(f"Batch size: {batch_size}")
print(f"Max epochs: {epochs}")

# Train the model
history = model.fit(
    X_train, y_train_cat,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_test, y_test_cat),
    callbacks=[early_stopping, reduce_lr, model_checkpoint],
    verbose=1
)

print("\nTraining completed!")

## Training History Visualization

Plot training and validation accuracy/loss over epochs.

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot accuracy
axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot loss
axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print(f"\nFinal Training Metrics:")
print(f"  Accuracy: {final_train_acc:.4f} ({final_train_acc*100:.2f}%)")
print(f"  Loss: {final_train_loss:.4f}")
print(f"\nFinal Validation Metrics:")
print(f"  Accuracy: {final_val_acc:.4f} ({final_val_acc*100:.2f}%)")
print(f"  Loss: {final_val_loss:.4f}")

## Evaluate Model

Make predictions and evaluate the model performance on the test set.

In [ ]:
# Make predictions
y_pred_proba = model.predict(X_test, verbose=0)  # Shape: (n_test_samples, n_classes)
y_pred_encoded = np.argmax(y_pred_proba, axis=1)  # Predicted class indices

print("Predictions shape:")
print(f"  y_pred_proba shape: {y_pred_proba.shape}")
print(f"  y_pred_encoded shape: {y_pred_encoded.shape}")

# Calculate accuracy
accuracy = accuracy_score(y_test_encoded, y_pred_encoded)
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

## Detailed Classification Report

In [ ]:
# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    y_test_encoded, 
    y_pred_encoded, 
    target_names=class_names,
    digits=3
))

## Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test_encoded, y_pred_encoded)

# Plot confusion matrix
plt.figure(figsize=(16, 14))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix - BiLSTM Workout Class Classifier', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Calculate per-class accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
print("\nPer-Class Accuracy:")
for i, cls_name in enumerate(class_names):
    print(f"  {cls_name}: {per_class_accuracy[i]:.4f} ({per_class_accuracy[i]*100:.2f}%)")

## Example Predictions

Show some example predictions with probabilities.

In [ ]:
# Show some example predictions
n_examples = 10
example_indices = np.random.choice(len(X_test), n_examples, replace=False)

print("Example Predictions:")
print("=" * 80)
for idx in example_indices:
    true_class_idx = y_test_encoded[idx]
    pred_class_idx = y_pred_encoded[idx]
    true_class = class_names[true_class_idx]
    pred_class = class_names[pred_class_idx]
    probabilities = y_pred_proba[idx]
    
    # Get top 3 predicted classes
    top3_indices = np.argsort(probabilities)[-3:][::-1]
    
    print(f"\nSample {idx}:")
    print(f"  True class: {true_class}")
    print(f"  Predicted class: {pred_class} {'✓' if true_class == pred_class else '✗'}")
    print(f"  Top 3 predictions:")
    for i, class_idx in enumerate(top3_indices):
        print(f"    {i+1}. {class_names[class_idx]}: {probabilities[class_idx]:.4f}")

## Save Model

Save the trained model and metadata for future use.

In [ ]:
# Save the final model
model_path = models_dir / 'bilstm_workout_classifier.keras'
model.save(model_path)
print(f"Model saved to: {model_path}")

# Save class names for reference
class_names_path = models_dir / 'class_names_bilstm.json'
with open(class_names_path, 'w') as f:
    json.dump(class_names.tolist(), f, indent=2)
print(f"Class names saved to: {class_names_path}")

# Save normalization parameters
normalization_params = {
    'mean': X_mean.tolist(),
    'std': X_std.tolist()
}
norm_params_path = models_dir / 'normalization_params_bilstm.json'
with open(norm_params_path, 'w') as f:
    json.dump(normalization_params, f, indent=2)
print(f"Normalization parameters saved to: {norm_params_path}")

# Save metadata about the model
model_metadata = {
    'model_type': 'BiLSTM',
    'n_classes': len(class_names),
    'n_timesteps': n_timesteps,
    'n_features': n_features,
    'sequence_shape': [n_timesteps, n_features],
    'class_names': class_names.tolist(),
    'test_accuracy': float(accuracy),
    'model_path': str(model_path),
    'class_names_path': str(class_names_path),
    'normalization_params_path': str(norm_params_path),
    'model_architecture': {
        'layers': [
            {'type': 'Bidirectional LSTM', 'units': 128, 'return_sequences': True},
            {'type': 'Bidirectional LSTM', 'units': 64, 'return_sequences': False},
            {'type': 'Dense', 'units': 64, 'activation': 'relu'},
            {'type': 'Dense', 'units': n_classes, 'activation': 'softmax'}
        ]
    }
}

metadata_path = models_dir / 'model_metadata_bilstm.json'
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)
print(f"Model metadata saved to: {metadata_path}")

## Prediction Function

Helper function to make predictions on new sequences.

In [ ]:
def predict_workout_class(sequence, model, class_names, normalization_params=None, return_proba=False):
    """
    Predict workout class for a sequence.
    
    Args:
        sequence: numpy array of shape (15, 12, 3) or (1, 15, 12, 3) or (15, 36) or (1, 15, 36)
        model: Trained BiLSTM model
        class_names: Array of class names
        normalization_params: Dict with 'mean' and 'std' for normalization (optional)
        return_proba: If True, return probabilities for all classes, else return class name
    
    Returns:
        If return_proba=True: probability array of shape (n_classes,)
        If return_proba=False: predicted class name (string)
    """
    # Reshape sequence to (n_samples, 15, 36)
    if sequence.ndim == 3:
        if sequence.shape[2] == 3:
            # Shape is (15, 12, 3) - reshape to (1, 15, 36)
            sequence = sequence.reshape(1, sequence.shape[0], -1)
        else:
            # Shape is (1, 15, 36) or (15, 36)
            if sequence.shape[0] == 15:
                sequence = sequence.reshape(1, *sequence.shape)
            else:
                sequence = sequence.reshape(sequence.shape[0], sequence.shape[1], -1)
    elif sequence.ndim == 4:
        # Shape is (n, 15, 12, 3) - reshape to (n, 15, 36)
        sequence = sequence.reshape(sequence.shape[0], sequence.shape[1], -1)
    
    # Handle NaN and infinite values
    sequence = np.nan_to_num(sequence, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Normalize if parameters provided
    if normalization_params is not None:
        mean = np.array(normalization_params['mean'])
        std = np.array(normalization_params['std'])
        sequence = (sequence - mean) / std
    
    # Get probabilities
    probabilities = model.predict(sequence, verbose=0)
    
    if return_proba:
        # Return probabilities
        return probabilities[0] if probabilities.shape[0] == 1 else probabilities
    else:
        # Return class name
        pred_class_idx = np.argmax(probabilities, axis=1)
        if len(pred_class_idx) == 1:
            return class_names[pred_class_idx[0]]
        else:
            return [class_names[idx] for idx in pred_class_idx]

# Test the prediction function
print("Testing prediction function:")
test_sequence = X[0]  # Original shape (15, 12, 3)
pred_class = predict_workout_class(
    test_sequence, 
    model, 
    class_names, 
    normalization_params={'mean': X_mean.tolist(), 'std': X_std.tolist()},
    return_proba=False
)
pred_proba = predict_workout_class(
    test_sequence, 
    model, 
    class_names,
    normalization_params={'mean': X_mean.tolist(), 'std': X_std.tolist()},
    return_proba=True
)

print(f"  Input shape: {test_sequence.shape}")
print(f"  Predicted class: {pred_class}")
print(f"  Probabilities shape: {pred_proba.shape}")
print(f"  Top 3 class probabilities:")
top3_indices = np.argsort(pred_proba)[-3:][::-1]
for i, idx in enumerate(top3_indices):
    print(f"    {i+1}. {class_names[idx]}: {pred_proba[idx]:.4f}")
print(f"  True class: {class_names[y_encoded[0]]}")